### 1. Answer the questions from the introduction

1) What is leave-one-out? Provide limitations and strengths.

LOOCV (Leave-One-Out Cross-Validation) - это крайний случай k-fold кросс-валидации, где количество разбиений (k) равно числу наблюдений (n).

На каждой итерации модель обучается на n-1 наблюдении и тестируется на одном оставшемся.
Сильные Стороны

Низкое Смещение (Low Bias): Оценка ошибки очень точно отражает истинную ошибку, так как обучающие наборы почти равны по размеру полному набору данных.

Детерминированность: Результат всегда один и тот же, поскольку нет случайного разбиения.

Ограничения

Высокая Стоимость: Требует n полных обучений модели. Чрезвычайно медленно при большом n.

Высокая Дисперсия: Оценки ошибок на разных итерациях сильно коррелируют, что может привести к высокой дисперсии итоговой оценки.

Чувствительность к Выбросам: Один выброс в тестовом наборе может сильно исказить оценку на данной итерации.

2) How do Grid Search, Randomized Grid Search, and Bayesian optimization work?

Методы Оптимизации Гиперпараметров

| Метод | Как работает | Принцип выбора | Плюсы | Минусы |
| :--- | :--- | :--- | :--- | :--- |
| **Grid Search** | Полный перебор всех возможных комбинаций значений, заданных в сетке. | **Систематический:** Проверяет каждую точку. | Гарантированно находит оптимум (в рамках сетки). | Очень медленно (экспоненциальный рост времени). |
| **Randomized Search** | Выбирает фиксированное число $N$ **случайных** комбинаций из заданных диапазонов. | **Случайный:** Эффективно исследует важные параметры. | Гораздо быстрее, легко контролировать время ($N$). | Может пропустить истинный оптимум. |
| **Bayesian Optimization** | Строит **суррогатную модель** (например, Гауссовский процесс) зависимости качества от параметров. | **Интеллектуальный/Последовательный:** Использует историю, чтобы выбрать самую перспективную следующую точку. | Требует наименьшее число запусков для нахождения оптимума. | Сложно, выполняется последовательно (плохо для параллелизации). |

3) Explain classification of feature selection methods. Explain how Pearson and Chi2 work. Explain how Lasso works. Explain what permutation significance is. Become familiar with SHAP.

Классификация Методов Отбора Признаков (Feature Selection)

Методы делятся на три категории по взаимодействию с моделью:

1.  **Фильтрующие методы (Filter Methods):** Оценивают признак **независимо от модели** (по статистике).
    * *Примеры:* **Корреляция Пирсона**, **Хи-квадрат ($\chi^2$)**.
    * *Плюсы:* Быстрые, устойчивы к переобучению.

2.  **Оборачивающие методы (Wrapper Methods):** Используют модель для оценки наборов признаков (перебор и переобучение).
    * *Примеры:* RFE, SFS.
    * *Плюсы:* Находят оптимальный набор для данной модели.

3.  **Встроенные методы (Embedded Methods) :** Отбор происходит **в процессе обучения** (регуляризация штрафует веса).
    * *Примеры:* **Lasso ($\text{L}_1$ регуляризация)**, Elastic Net.
    * *Плюсы:* Эффективны, учитывают взаимодействие признаков.

Примеры Методов

#### Корреляция Пирсона (Pearson)
Измеряет **линейную зависимость** между двумя непрерывными переменными ($X$ и $Y$). Выбираются признаки с высоким абсолютным значением $|r|$.

#### Хи-квадрат ($\chi^2$, Chi-squared)
Оценивает взаимосвязь между **категориальным признаком** ($X$) и **категориальной целевой переменной** ($Y$). Большое значение $\chi^2$ указывает на сильную статистическую связь (важность признака).

#### Lasso ($\text{L}_1$ Регуляризация)
Добавляет штраф, пропорциональный $\sum_{i=1}^{n}|\beta_i|$ (сумме абсолютных значений весов). Этот штраф **обнуляет веса** ($\beta_i$) наименее важных признаков, выполняя автоматический отбор.

#### Permutation Significance (Значимость перестановки)

* Метод оценки важности признака после того, как модель обучена.
* Измеряет, насколько сильно падает качество модели, если случайно перемешать значения одного признака в тестовом наборе.
* Сильное падение качества = важный признак. Слабое изменение = неважный признак.

#### SHAP (SHapley Additive exPlanations)

* Мощный метод объяснимого ИИ (XAI), основанный на теории игр. Объясняет предсказание любой модели (от регрессии до нейронных сетей).
* Вычисляет вклад каждого признака в итоговое предсказание для конкретного наблюдения (смещение результата от базового среднего).
* Позволяет объяснить, почему модель дала именно такой результат для одной точки (локальная интерпретация) и оценить общую важность признаков (глобальная интерпретация).

### 2. Introduction — do all the preprocessing from the previous lesson

In [107]:
import pandas as pd
import numpy as np
import sklearn
import lightgbm
import scipy
import statsmodels
import re
from sklearn.preprocessing import PolynomialFeatures, MultiLabelBinarizer
from collections import Counter
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import KFold, GroupKFold, StratifiedKFold, TimeSeriesSplit
from collections import defaultdict
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.model_selection import train_test_split
import shap
from sklearn.inspection import permutation_importance
from sklearn.metrics import make_scorer
from scipy.stats import uniform
import optuna

In [108]:
test= pd.read_json("./data/test.json", orient="columns")
train = pd.read_json("./data/train.json", orient="columns")

In [109]:
train.head(3)

,bathrooms,bedrooms,building_id,created,description,display_address,features,latitude,listing_id,longitude,manager_id,photos,price,street_address,interest_level
4,1.0,1,8579a0b0d54db803821a35a4a615e97a,2016-06-16 05:55:27,Spacious 1 Bedroom 1 Bathroom in Williamsburg!...,145 Borinquen Place,"[Dining Room, Pre-War, Laundry in Building, Di...",40.7108,7170325,-73.9539,a10db4590843d78c784171a107bdacb4,[https://photos.renthop.com/2/7170325_3bb5ac84...,2400,145 Borinquen Place,medium
6,1.0,2,b8e75fc949a6cd8225b455648a951712,2016-06-01 05:44:33,BRAND NEW GUT RENOVATED TRUE 2 BEDROOMFind you...,East 44th,"[Doorman, Elevator, Laundry in Building, Dishw...",40.7513,7092344,-73.9722,955db33477af4f40004820b4aed804a0,[https://photos.renthop.com/2/7092344_7663c19a...,3800,230 East 44th,low
9,1.0,2,cd759a988b8f23924b5a2058d5ab2b49,2016-06-14 15:19:59,**FLEX 2 BEDROOM WITH FULL PRESSURIZED WALL**L...,East 56th Street,"[Doorman, Elevator, Laundry in Building, Laund...",40.7575,7158677,-73.9625,c8b10a317b766204f08e613cef4ce7a0,[https://photos.renthop.com/2/7158677_c897a134...,3495,405 East 56th Street,medium


In [110]:
test.head(3)

,bathrooms,bedrooms,building_id,created,description,display_address,features,latitude,listing_id,longitude,manager_id,photos,price,street_address
0,1.0,1,79780be1514f645d7e6be99a3de696c5,2016-06-11 05:29:41,Large with awesome terrace--accessible via bed...,Suffolk Street,"[Elevator, Laundry in Building, Laundry in Uni...",40.7185,7142618,-73.9865,b1b1852c416d78d7765d746cb1b8921f,[https://photos.renthop.com/2/7142618_1c45a2c8...,2950,99 Suffolk Street
1,1.0,2,0,2016-06-24 06:36:34,Prime Soho - between Bleecker and Houston - Ne...,Thompson Street,"[Pre-War, Dogs Allowed, Cats Allowed]",40.7278,7210040,-74.0000,d0b5648017832b2427eeb9956d966a14,[https://photos.renthop.com/2/7210040_d824cc71...,2850,176 Thompson Street
2,1.0,0,0,2016-06-17 01:23:39,Spacious studio in Prime Location. Cleanbuildi...,Sullivan Street,"[Pre-War, Dogs Allowed, Cats Allowed]",40.7260,7174566,-74.0026,e6472c7237327dd3903b3d6f6a94515a,[https://photos.renthop.com/2/7174566_ba3a35c5...,2295,115 Sullivan Street


Очищение от выбросов и препроцессинг целевой переменной (Interest Level)

high → 2, medium → 1, low → 0

In [111]:
ulimit = np.percentile(train.price.values, 99)
train = train[(train['price'] > 200) & (train['price'] < ulimit)].reset_index(drop=True)

train['interest_level'] = train['interest_level'].map({'low': 0, 'medium': 1, 'high': 2})
train['interest_level']

,interest_level
0,1
1,0
2,1
3,1
4,0
...,...
48843,0
48844,1
48845,1
48846,1


In [112]:
feature_list = [
    'Elevator', 'HardwoodFloors', 'CatsAllowed', 'DogsAllowed', 'Doorman', 'Dishwasher', 'NoFee', 'LaundryinBuilding',
    'FitnessCenter', 'Pre-War', 'LaundryinUnit', 'RoofDeck', 'OutdoorSpace', 'DiningRoom', 'HighSpeedInternet', 'Balcony',
    'SwimmingPool', 'LaundryInBuilding', 'NewConstruction', 'Terrace'
]

In [113]:
def clean_feature(feature):
    return re.sub(r"[^a-zA-Z0-9]", "", feature).lower()

def extract_features(df):
    df_feat = pd.DataFrame(index=df.index)
    for f in feature_list: df_feat[f] = 0
    for idx, row in df.iterrows():
        if isinstance(row.get("features"), list):
            feats = set([clean_feature(f) for f in row["features"]])
            for f in feature_list:
                if clean_feature(f) in feats: df_feat.at[idx, f] = 1
    return df_feat

train_feat = extract_features(train)
test_feat  = extract_features(test)

for col in ["bathrooms", "bedrooms", "interest_level", "price"]:
    train_feat[col] = train[col]

y_all = train_feat["price"]
X_all = train_feat.drop(columns=["price"])
data_all = pd.concat([X_all.reset_index(drop=True), y_all.reset_index(drop=True)], axis=1)

In [114]:
X_all

,Elevator,HardwoodFloors,CatsAllowed,DogsAllowed,Doorman,Dishwasher,NoFee,LaundryinBuilding,FitnessCenter,Pre-War,...,DiningRoom,HighSpeedInternet,Balcony,SwimmingPool,LaundryInBuilding,NewConstruction,Terrace,bathrooms,bedrooms,interest_level
0,0,1,1,1,0,1,0,1,0,1,...,1,0,0,0,1,0,0,1.0,1,1
1,1,1,0,0,1,1,1,1,0,0,...,0,0,0,0,1,0,0,1.0,2,0
2,1,1,0,0,1,1,0,1,0,0,...,0,0,0,0,1,0,0,1.0,2,1
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1.5,3,1
4,1,0,0,0,1,0,0,1,1,0,...,0,0,0,0,1,0,0,1.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48843,1,1,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,1.0,3,0
48844,1,0,1,1,1,0,1,1,0,0,...,0,0,0,0,1,0,0,1.0,2,1
48845,1,1,1,1,0,1,1,1,0,1,...,1,0,0,0,1,0,0,1.0,1,1
48846,0,0,0,0,0,1,1,0,0,1,...,0,0,0,0,0,0,0,1.0,2,1


In [115]:
y_all

,price
0,2400
1,3800
2,3495
3,3000
4,2795
...,...
48843,2800
48844,2395
48845,1850
48846,4195


### 3. Implement the next methods:

Split data into 2 parts randomly with parameter test_size (ratio from 0 to 1), return training and test samples.

(Разделить данные на 2 части случайным образом с параметром test_size (соотношение от 0 до 1), вернуть обучающую и тестовую выборки.)

In [116]:
def prepare_return(df_train, df_test, df_valid=None):
    """Вспомогательная функция для отделения X, y и удаления временных колонок."""
    drop_cols = ['price']

    X_train = df_train.drop(columns=drop_cols, errors='ignore')
    y_train = df_train['price']
    X_test = df_test.drop(columns=drop_cols, errors='ignore')
    y_test = df_test['price']

    if df_valid is not None:
        X_valid = df_valid.drop(columns=drop_cols, errors='ignore')
        y_valid = df_valid['price']
        return X_train, y_train, X_valid, y_valid, X_test, y_test

    return X_train, y_train, X_test, y_test

In [117]:
def random_split_two(data, test_size=0.2, random_state=42):
    if random_state: np.random.seed(random_state)
    indices = np.random.permutation(len(data))
    split_point = int((1 - test_size) * len(data))

    train_idx, test_idx = indices[:split_point], indices[split_point:]
    return prepare_return(data.iloc[train_idx], data.iloc[test_idx])

Randomly split data into 3 parts with parameters validation_size and test_size, return train, validation and test samples.

(Случайным образом разделить данные на 3 части с параметрами validation_size и test_size, вернуть обучающую, проверочную и тестовую выборки.)

Случайное разделение данных 60/20/20

In [118]:
def random_split_three(X, y):
    np.random.seed(42)
    indices = np.random.permutation(len(X))
    n = len(X)
    valid_split = int(0.6 * n)
    test_split = int(0.8 * n)

    idx_train = X.iloc[indices[:valid_split]].index
    idx_valid = X.iloc[indices[valid_split:test_split]].index
    idx_test  = X.iloc[indices[test_split:]].index

    return (X.loc[idx_train], y.loc[idx_train],
            X.loc[idx_valid], y.loc[idx_valid],
            X.loc[idx_test],  y.loc[idx_test])

X_train, y_train, X_valid, y_val, X_test, y_test = random_split_three(X_all, y_all)

Split data into 2 parts with parameter date_split, return train and test samples split by date_split param.

(Разделите данные на 2 части с помощью параметра date_split, верните обучающую и тестовую выборки, разделенные по параметру date_split.)

In [119]:
def date_split_two(data, date_split, date_col="created"):
    data_sorted = data.sort_values(by=date_col)
    train = data_sorted[data_sorted[date_col] < date_split]
    test = data_sorted[data_sorted[date_col] >= date_split]
    return prepare_return(train, test)

Split data into 3 parts with parameters validation_date and test_date, return train, validation and test samples split by input params.

(Разделите данные на 3 части с параметрами validation_date и test_date, верните тренировочный, проверочный и тестовый образец, разделенные по входным параметрам.)

In [120]:
def date_split_three(data, validation_date, test_date, date_col="created"):
    data_sorted = data.sort_values(by=date_col)
    train = data_sorted[data_sorted[date_col] < validation_date]
    valid = data_sorted[(data_sorted[date_col] >= validation_date) & (data_sorted[date_col] < test_date)]
    test = data_sorted[data_sorted[date_col] >= test_date]
    return prepare_return(train, test, valid)

Make split procedure determenistic. What does it mean?

Сделать процедуру разделения детерминированной. Что это значит?

Сделать процедуру разделения **детерминированной** означает, что при каждом запуске будем получать одинаковый результат разбиения.

Это достигается за счет фиксации внутреннего состояния генератора псевдослучайных чисел.

В большинстве библиотек машинного обучения (например, в функции train_test_split из Scikit-learn) процесс разбиения включает элемент случайности, так как он случайно перемешивает данные перед делением.

Чтобы сделать этот процесс повторяемым, используется параметр random_state:

Это начальное число, или "семя" (seed), которое используется для инициализации генератора псевдослучайных чисел.

Воспроизводимость (Reproducibility): позволяет повторить эксперимент и получить те же результаты. Если не зафиксировать random_state, изменение результатов обучения может быть связано как с изменениями в коде, так и со случайным изменением разбиения данных.

Сравнение Моделей: Чтобы честно сравнить две разные модели или два набора гиперпараметров, они должны быть обучены и протестированы на одних и тех же обучающих и тестовых данных.

Отладка (Debugging): Если в коде возникла ошибка, детерминированное разбиение позволяет вам изолировать и найти ошибку, так как она будет проявляться одинаково при каждом запуске.

### 4. Implement the next cross-validation methods:

K-Fold, where k is the input parameter, returns a list of train and test indices.

(K-Fold, где k — входной параметр, возвращает список обучающих и тестовых индексов.)

In [121]:
def k_fold_split(df: pd.DataFrame, k: int):
    n = len(df)
    indices = np.arange(n)

    fold_sizes = np.full(k, n // k, dtype=int)
    fold_sizes[: n % k] += 1

    current = 0
    splits = []
    for fold_size in fold_sizes:
        start, stop = current, current + fold_size

        test_idx = indices[start:stop]
        train_idx = np.concatenate((indices[:start], indices[stop:]))

        splits.append((train_idx, test_idx))
        current = stop

    return splits

Grouped K-Fold, where k and group_field are input parameters, returns list of train and test indices.

(Группированный K-фолд, где k и group_field являются входными параметрами, возвращает список обучающих и тестовых индексов.)

In [122]:
def group_k_fold_split(df: pd.DataFrame, k: int, group_field: str):
    unique_groups = df[group_field].unique()
    np.random.shuffle(unique_groups)

    group_folds = np.array_split(unique_groups, k)

    splits = []
    for i in range(k):
        test_groups = group_folds[i]

        test_idx = df[df[group_field].isin(test_groups)].index.values
        train_idx = df[~df[group_field].isin(test_groups)].index.values

        splits.append((train_idx, test_idx))

    return splits

Stratified K-fold, where k and stratify_field are input parameters, returns list of train and test indices.

(Стратифицированное K-сложение, где k и stratify_field являются входными параметрами, возвращает список обучающих и тестовых индексов.)

In [123]:
def stratified_k_fold_split(df: pd.DataFrame, k: int, stratify_field: str):
    label_to_indices = defaultdict(list)
    for idx in df.index:
        label_to_indices[df.loc[idx, stratify_field]].append(idx)

    folds = [[] for _ in range(k)]
    for indices in label_to_indices.values():
        shuffled_indices = np.random.permutation(indices)
        split_indices = np.array_split(shuffled_indices, k)

        for fold_idx in range(k):
            folds[fold_idx].extend(split_indices[fold_idx])

    splits = []
    for i in range(k):
        test_idx = np.array(folds[i])
        train_idx_list = []
        for j in range(k):
            if j != i:
                train_idx_list.extend(folds[j])
        train_idx = np.array(train_idx_list)

        splits.append((train_idx, test_idx))

    return splits

Time series split, where k and date_field are input parameters, returns list of train and test indices.

(Разделение временного ряда, где k и date_field являются входными параметрами, возвращает список обучающих и тестовых индексов.)

In [124]:
def time_series_split(df: pd.DataFrame, k: int, date_field: str):
    sorted_indices = df.sort_values(by=date_field).index.values
    n_samples = len(sorted_indices)

    test_size = n_samples // (k + 1)

    splits = []
    for i in range(1, k + 1):
        train_end_idx = i * test_size
        test_start_idx = train_end_idx

        train_idx = sorted_indices[:train_end_idx]
        test_idx = sorted_indices[test_start_idx:test_start_idx + test_size] # Тест всегда фиксированного размера

        if len(test_idx) == 0:
            break

        splits.append((train_idx, test_idx))

    return splits

GroupKFold

group_k_fold_split оперирует на уровне уникальных групп (unique_groups), делит эти группы на k частей, и затем назначает все строки, принадлежащие тестовым группам, в тестовую выборку (test_idx), гарантируя, что одна группа не будет разделена между обучающей и тестовой выборкой.

StratifiedKFold

stratified_k_fold_split использует словарь label_to_indices, чтобы сначала собрать все индексы, принадлежащие каждому классу (stratify_field). Затем она равномерно распределяет эти индексы между k фолдами, сохраняя пропорциональное соотношение классов в каждой тестовой и обучающей выборке.

### 5. Cross-validation comparison

Подготавливаю тестовый датасет для проверки разных стратегий разбиения на фолды: KFold, TimeSeriesSplit, StratifiedKFold, GroupKFold.

In [125]:
N_TEST = 40
K = 5
np.random.seed(42)

test_df = train_feat.head(N_TEST).copy()
test_df['created'] = train['created'].head(N_TEST).values
test_df['group_id'] = test_df['price'].apply(lambda x: f'P_{int(x) // 1000}')
test_df['bedrooms_cat'] = test_df['bedrooms'].astype(str)

test_df_kf = test_df.reset_index(drop=True)
test_df_orig = test_df.copy()

Сравнение самописных функций с scikit-learn

KFold

In [126]:
kf = KFold(n_splits=5)
sklearn_splits = [(train, test) for train, test in kf.split(train_feat, train_feat['price'])]
sklearn_splits

[(array([ 9770,  9771,  9772, ..., 48845, 48846, 48847]),
  array([   0,    1,    2, ..., 9767, 9768, 9769])),
 (array([    0,     1,     2, ..., 48845, 48846, 48847]),
  array([ 9770,  9771,  9772, ..., 19537, 19538, 19539])),
 (array([    0,     1,     2, ..., 48845, 48846, 48847]),
  array([19540, 19541, 19542, ..., 29307, 29308, 29309])),
 (array([    0,     1,     2, ..., 48845, 48846, 48847]),
  array([29310, 29311, 29312, ..., 39076, 39077, 39078])),
 (array([    0,     1,     2, ..., 39076, 39077, 39078]),
  array([39079, 39080, 39081, ..., 48845, 48846, 48847]))]

In [127]:
k_fold_split(train_feat,k=5)

[(array([ 9770,  9771,  9772, ..., 48845, 48846, 48847]),
  array([   0,    1,    2, ..., 9767, 9768, 9769])),
 (array([    0,     1,     2, ..., 48845, 48846, 48847]),
  array([ 9770,  9771,  9772, ..., 19537, 19538, 19539])),
 (array([    0,     1,     2, ..., 48845, 48846, 48847]),
  array([19540, 19541, 19542, ..., 29307, 29308, 29309])),
 (array([    0,     1,     2, ..., 48845, 48846, 48847]),
  array([29310, 29311, 29312, ..., 39076, 39077, 39078])),
 (array([    0,     1,     2, ..., 39076, 39077, 39078]),
  array([39079, 39080, 39081, ..., 48845, 48846, 48847]))]

GroupKFold

In [128]:
gkf_lib = GroupKFold(n_splits=K)
skf_splits = gkf_lib.split(test_df_kf, groups=test_df_kf['group_id'])

my_splits = group_k_fold_split(test_df_orig, K, 'group_id')

for i, ((my_train, my_test), (sk_train, sk_test)) in enumerate(zip(my_splits, skf_splits)):

    sk_train_groups = set(test_df_kf.iloc[sk_train]['group_id'])
    sk_test_groups = set(test_df_kf.iloc[sk_test]['group_id'])

    my_train_groups = set(test_df_orig.loc[my_train]['group_id'])
    my_test_groups = set(test_df_orig.loc[my_test]['group_id'])

    print(f"\nFold {i+1}")
    print(f"My: {len(my_train)} train, {len(my_test)} test, group intersection={len(my_train_groups & my_test_groups)}")
    print(f"Sklearn: {len(sk_train)} train, {len(sk_test)} test, group intersection={len(sk_train_groups & sk_test_groups)}")


Fold 1
My: 14 train, 26 test, group intersection=0
Sklearn: 26 train, 14 test, group intersection=0

Fold 2
My: 34 train, 6 test, group intersection=0
Sklearn: 28 train, 12 test, group intersection=0

Fold 3
My: 37 train, 3 test, group intersection=0
Sklearn: 35 train, 5 test, group intersection=0

Fold 4
My: 38 train, 2 test, group intersection=0
Sklearn: 35 train, 5 test, group intersection=0

Fold 5
My: 37 train, 3 test, group intersection=0
Sklearn: 36 train, 4 test, group intersection=0


StratifiedKFold

In [129]:
stratify_field = 'bedrooms_cat'

skf_lib = StratifiedKFold(n_splits=K, shuffle=True, random_state=42)
lib_splits = list(skf_lib.split(test_df_kf, test_df_kf[stratify_field]))

custom_splits = stratified_k_fold_split(test_df_orig, K, stratify_field)

for i, ((custom_train, custom_test), (lib_train, lib_test)) in enumerate(zip(custom_splits, lib_splits)):

    custom_test_classes = test_df_orig.loc[custom_test, stratify_field]
    custom_train_classes = test_df_orig.loc[custom_train, stratify_field]

    lib_test_classes = test_df_kf.iloc[lib_test][stratify_field]
    lib_train_classes = test_df_kf.iloc[lib_train][stratify_field]

    custom_test_counts = Counter(custom_test_classes)
    custom_train_counts = Counter(custom_train_classes)

    lib_test_counts = Counter(lib_test_classes)
    lib_train_counts = Counter(lib_train_classes)

    print(f"\nFold {i+1}")
    print(f"My implementation - Test Counts: {custom_test_counts}")
    print(f"Sklearn          - Test Counts: {lib_test_counts}")

    total_test = len(custom_test)
    if total_test > 0:
        custom_test_proportions = {k: f"{v / total_test * 100:.1f}%" for k, v in custom_test_counts.items()}
        lib_test_proportions = {k: f"{v / total_test * 100:.1f}%" for k, v in lib_test_counts.items()}

        print(f"My implementation - Proportions: {custom_test_proportions}")
        print(f"Sklearn          - Proportions: {lib_test_proportions}")


Fold 1
My implementation - Test Counts: Counter({'2': 3, '1': 2, '3': 2, '0': 2, '4': 1})
Sklearn          - Test Counts: Counter({'2': 3, '1': 2, '0': 2, '3': 1})
My implementation - Proportions: {'1': '20.0%', '2': '30.0%', '3': '20.0%', '0': '20.0%', '4': '10.0%'}
Sklearn          - Proportions: {'2': '30.0%', '1': '20.0%', '3': '10.0%', '0': '20.0%'}

Fold 2
My implementation - Test Counts: Counter({'1': 2, '2': 2, '3': 2, '0': 2})
Sklearn          - Test Counts: Counter({'3': 2, '0': 2, '1': 2, '2': 2})
My implementation - Proportions: {'1': '25.0%', '2': '25.0%', '3': '25.0%', '0': '25.0%'}
Sklearn          - Proportions: {'3': '25.0%', '0': '25.0%', '1': '25.0%', '2': '25.0%'}

Fold 3
My implementation - Test Counts: Counter({'1': 2, '2': 2, '3': 2, '0': 2})
Sklearn          - Test Counts: Counter({'2': 2, '0': 2, '1': 2, '3': 2})
My implementation - Proportions: {'1': '25.0%', '2': '25.0%', '3': '25.0%', '0': '25.0%'}
Sklearn          - Proportions: {'2': '25.0%', '0': '25.0%'

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


TimeSeriesSplit

In [144]:
date_field = 'created'

tscv = TimeSeriesSplit(n_splits=K)
sklearn_splits = tscv.split(test_df_kf)

my_splits = time_series_split(test_df_orig, K, date_field)

for i, (my_fold, sk_fold) in enumerate(zip(my_splits, sklearn_splits), 1):
    my_train_idx, my_test_idx = my_fold
    sk_train_idx, sk_test_idx = sk_fold

    sk_train_dates = test_df_kf.iloc[sk_train_idx][date_field]
    sk_test_dates = test_df_kf.iloc[sk_test_idx][date_field]

    my_train_dates = test_df_orig.loc[my_train_idx][date_field]
    my_test_dates = test_df_orig.loc[my_test_idx][date_field]

    my_order_ok = my_train_dates.max() < my_test_dates.min()
    sk_order_ok = sk_train_dates.max() < sk_test_dates.min()

    print(f"\nFold {i}")
    print(f"My:  Train size {len(my_train_idx)}, Test size {len(my_test_idx)}, Chronology OK: {my_order_ok}")
    print(f"Sklearn: Train size {len(sk_train_idx)}, Test size {len(sk_test_idx)}, Chronology OK: {sk_order_ok}")


Fold 1
My:  Train size 6, Test size 6, Chronology OK: True
Sklearn: Train size 10, Test size 6, Chronology OK: False

Fold 2
My:  Train size 12, Test size 6, Chronology OK: True
Sklearn: Train size 16, Test size 6, Chronology OK: False

Fold 3
My:  Train size 18, Test size 6, Chronology OK: True
Sklearn: Train size 22, Test size 6, Chronology OK: False

Fold 4
My:  Train size 24, Test size 6, Chronology OK: True
Sklearn: Train size 28, Test size 6, Chronology OK: False

Fold 5
My:  Train size 30, Test size 6, Chronology OK: True
Sklearn: Train size 34, Test size 6, Chronology OK: False


1. Реализованы все механизмы разделения

В коде присутствуют и корректно реализованы все четыре требуемых типа кросс-валидации:

  k_fold_split: Обычный K-Fold.

  group_k_fold_split: Группированный K-Fold (Grouped K-Fold).

  stratified_k_fold_split: Стратифицированный K-Fold (Stratified K-Fold).

  time_series_split: Разделение временного ряда (Time Series Split).

Каждая функция принимает требуемые входные параметры (k, group_field, stratify_field, date_field) и возвращает список пар индексов (train_idx, test_idx).

2. Соблюдение временной зависимости

Функция time_series_split использует алгоритм Walk-Forward Validation (последовательная кросс-валидация) и гарантирует, что принцип зависимости от времени соблюдается:

Для каждого генерируемого среза данных:
Max(Train Timestamp)<Min(Test Timestamp)

3. Детерминированное разделение (Seed)

В коде установлена фиксация генератора случайных чисел (np.random.seed(42)).

Это обеспечивает детерминированность (воспроизводимость) всех процессов, связанных со случайностью:

  Перемешивание групп в group_k_fold_split.

  Перемешивание индексов внутри классов в stratified_k_fold_split.

Таким образом, при повторном запуске кода индексы, возвращаемые этими функциями, будут всегда одинаковыми.

Бинирование (Binning) цены: Поскольку price — это непрерывная переменная, ее нельзя напрямую использовать для стратификации.

Создаю новую категориальную колонку (например, price_bin), разбив price на 5-10 интервалов (например, с помощью pd.cut или KBinsDiscretizer).

Передаю новую колонку в функцию: stratified_k_fold_split(df, k=5, stratify_field='price_bin').

Этот подход позволяет применить StratifiedKFold к непрерывной переменной, гарантируя, что редкие, дорогие дома ("long tail") будут равномерно распределены по всем фолдам для стабильной оценки модели.

In [131]:
N_BINS = 5
train_feat['price_bin'] = pd.cut(
    train_feat['price'],
    bins=N_BINS,
    labels=False,
    include_lowest=True
)

print(f"Распределение 'price_bin' по {N_BINS} интервалам:")
print(train_feat['price_bin'].value_counts(normalize=True).sort_index().to_markdown())

K = 5
print("Стратификация по цене (Stratified K-Fold on 'price_bin')")

price_stratified_splits = stratified_k_fold_split(train_feat, k=K, stratify_field='price_bin')

first_test_indices = price_stratified_splits[0][1]
test_df_sample = train_feat.loc[first_test_indices]

print(f"\nРазмер первого тестового фолда: {len(first_test_indices)} строк")
print("Распределение 'price_bin' в первом тестовом фолде")
print(test_df_sample['price_bin'].value_counts(normalize=True).sort_index().to_markdown(floatfmt=".4f"))

initial_ratio = train_feat['price_bin'].value_counts(normalize=True).iloc[0]
test_ratio = test_df_sample['price_bin'].value_counts(normalize=True).iloc[0]

print(f"\nДоля класса 0 в полном наборе данных: {initial_ratio:.4f}")
print(f"Доля класса 0 в тестовом фолде: {test_ratio:.4f}")
print("Это подтверждает, что соотношение классов сохранено.")

Распределение 'price_bin' по 5 интервалам:
|   price_bin |   proportion |
|------------:|-------------:|
|           0 |   0.429496   |
|           1 |   0.463581   |
|           2 |   0.0835653  |
|           3 |   0.0160907  |
|           4 |   0.00726744 |
Стратификация по цене (Stratified K-Fold on 'price_bin')

Размер первого тестового фолда: 9771 строк
Распределение 'price_bin' в первом тестовом фолде
|   price_bin |   proportion |
|------------:|-------------:|
|           0 |       0.4294 |
|           1 |       0.4635 |
|           2 |       0.0836 |
|           3 |       0.0162 |
|           4 |       0.0073 |

Доля класса 0 в полном наборе данных: 0.4636
Доля класса 0 в тестовом фолде: 0.4635
Это подтверждает, что соотношение классов сохранено.


### 6. Feature Selection

Fit a Lasso regression model with normalized features. Use your method for splitting samples into 3 parts by field created with 60/20/20 ratio — train/validation/test.

(Подберите модель регрессии лассо с нормализованными признаками. Используйте свой метод для разделения выборок на 3 части по полям, созданным в соотношении 60/20/20 — обучение/проверка/тест.)

In [132]:
X_train_num = X_train.select_dtypes(include=[np.number])
X_valid_num = X_valid.select_dtypes(include=[np.number])
X_test_num  = X_test.select_dtypes(include=[np.number])

scaler = MinMaxScaler()
X_train_s = scaler.fit_transform(X_train_num)
X_valid_s = scaler.transform(X_valid_num)
X_test_s  = scaler.transform(X_test_num)

In [133]:
lasso_full = Lasso(alpha=0.1, random_state=21, max_iter=20000)
lasso_full.fit(X_train_s, y_train)

importance = pd.Series(np.abs(lasso_full.coef_), index=X_train_num.columns)
top10 = importance.sort_values(ascending=False).head(20)
zero_feats = importance[importance == 0]

print("ID\tName\t\tImportance")
for col, val in top10.items():
    print(f"{X_train_num.columns.get_loc(col):2d}\t{col:20s}\t{val:.6f}")
for col in zero_feats.index[:5]:
    print(f"{X_train_num.columns.get_loc(col):2d}\t{col:20s}\t0.000000")
if len(zero_feats) > 5:
    print("...\t...\t...")

ID	Name		Importance
20	bathrooms           	14709.285392
21	bedrooms            	3877.456983
22	interest_level      	895.781287
 4	Doorman             	602.666191
10	LaundryinUnit       	438.871506
19	Terrace             	202.262363
 8	FitnessCenter       	202.075720
 0	Elevator            	149.015367
13	DiningRoom          	144.375695
14	HighSpeedInternet   	136.559139
18	NewConstruction     	134.883760
 9	Pre-War             	133.767406
 3	DogsAllowed         	121.054234
17	LaundryInBuilding   	116.410508
 5	Dishwasher          	111.690569
 6	NoFee               	97.628865
 7	LaundryinBuilding   	95.785796
 1	HardwoodFloors      	75.883195
12	OutdoorSpace        	71.218335
11	RoofDeck            	70.078524
15	Balcony             	0.000000


In [134]:
def evaluate(model, X_tr, X_val, X_te, y_tr, y_val, y_te, name):
    return {
        "model": name,
        "mae": [
            mean_absolute_error(y_tr, model.predict(X_tr)),
            mean_absolute_error(y_val, model.predict(X_val)),
            mean_absolute_error(y_te, model.predict(X_te)),
        ],
        "rmse": [
            np.sqrt(mean_squared_error(y_tr, model.predict(X_tr))),
            np.sqrt(mean_squared_error(y_val, model.predict(X_val))),
            np.sqrt(mean_squared_error(y_te, model.predict(X_te))),
        ],
        "r2": [
            r2_score(y_tr, model.predict(X_tr)),
            r2_score(y_val, model.predict(X_val)),
            r2_score(y_te, model.predict(X_te)),
        ]
    }

In [135]:
def metric_table(metrics, key):
    print("model\ttrain\tvalid\ttest")
    for m in metrics:
        tr, va, te = m[key]
        print(f"{m['model']}\t{tr:.6f}\t{va:.6f}\t{te:.6f}")

metrics = []

In [136]:
# Lasso full
metrics.append(evaluate(lasso_full, X_train_s, X_valid_s, X_test_s, y_train, y_val, y_test, "Lasso MinMaxScaler"))

# Lasso top10
cols = top10.index.tolist()
idxs = [X_train_num.columns.get_loc(c) for c in cols]
lasso_top10 = Lasso(alpha=0.1, random_state=21, max_iter=20000)
lasso_top10.fit(X_train_s[:, idxs], y_train)
metrics.append(evaluate(lasso_top10, X_train_s[:, idxs], X_valid_s[:, idxs], X_test_s[:, idxs], y_train, y_val, y_test, "Lasso top10 MinMaxScaler"))

print("MAE")
metric_table(metrics, "mae")
print("\nRMSE")
metric_table(metrics, "rmse")
print("\nR²")
metric_table(metrics, "r2")

MAE
model	train	valid	test
Lasso MinMaxScaler	694.229071	685.342591	681.668548
Lasso top10 MinMaxScaler	694.307082	685.183373	681.711667

RMSE
model	train	valid	test
Lasso MinMaxScaler	1008.221322	973.255097	987.718662
Lasso top10 MinMaxScaler	1008.322721	973.262092	987.811224

R²
model	train	valid	test
Lasso MinMaxScaler	0.604320	0.617410	0.610598
Lasso top10 MinMaxScaler	0.604240	0.617405	0.610525


Permutation importance

In [137]:
def mean_absolute_percentage_error(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    non_zero_idx = y_true != 0
    if not np.any(non_zero_idx):
        return np.nan
    return np.mean(np.abs((y_true[non_zero_idx] - y_pred[non_zero_idx]) / y_true[non_zero_idx]))

neg_mape_scorer = make_scorer(mean_absolute_percentage_error, greater_is_better=False)

perm = permutation_importance(lasso_full, X_valid_s, y_val, scoring=neg_mape_scorer, n_repeats=10, random_state=21)
perm_mean = pd.Series(perm.importances_mean, index=X_train_num.columns)
perm_std  = pd.Series(perm.importances_std,  index=X_train_num.columns)

perm_top10 = perm_mean.abs().sort_values(ascending=False).head(10).index.tolist()
perm_idxs = [X_train_num.columns.get_loc(c) for c in perm_top10]

lasso_perm = Lasso(alpha=0.1, random_state=21, max_iter=20000)
lasso_perm.fit(X_train_s[:, perm_idxs], y_train)
metrics.append(evaluate(lasso_perm, X_train_s[:, perm_idxs], X_valid_s[:, perm_idxs], X_test_s[:, perm_idxs], y_train, y_val, y_test, "Lasso permutation MinMaxScaler"))

print("\nMAE")
metric_table(metrics, "mae")
print("\nRMSE")
metric_table(metrics, "rmse")
print("\nR²")
metric_table(metrics, "r2")


MAE
model	train	valid	test
Lasso MinMaxScaler	694.229071	685.342591	681.668548
Lasso top10 MinMaxScaler	694.307082	685.183373	681.711667
Lasso permutation MinMaxScaler	699.787075	692.365413	685.028143

RMSE
model	train	valid	test
Lasso MinMaxScaler	1008.221322	973.255097	987.718662
Lasso top10 MinMaxScaler	1008.322721	973.262092	987.811224
Lasso permutation MinMaxScaler	1017.555679	983.920945	994.276833

R²
model	train	valid	test
Lasso MinMaxScaler	0.604320	0.617410	0.610598
Lasso top10 MinMaxScaler	0.604240	0.617405	0.610525
Lasso permutation MinMaxScaler	0.596959	0.608979	0.605410


SHAP

In [138]:
explainer = shap.LinearExplainer(lasso_full, X_train_s, feature_names=X_train_num.columns)
shap_values = explainer.shap_values(X_train_s)
shap_importances = np.abs(shap_values).mean(axis=0)
shap_imp_df = pd.DataFrame({"feature": X_train_num.columns, "shap_value": shap_importances}).sort_values("shap_value", ascending=False)

shap_table = shap_imp_df.head(10).copy()
shap_table.reset_index(drop=True, inplace=True)
shap_table.index.name = 'Rank'
shap_table.rename(columns={'feature': 'Feature', 'shap_value': 'SHAP Importance'}, inplace=True)

display(shap_table)

metrics_table = pd.DataFrame(columns=["Model", "MAE_Train", "MAE_Valid", "MAE_Test",
                                      "RMSE_Train", "RMSE_Valid", "RMSE_Test",
                                      "R2_Train", "R2_Valid", "R2_Test"])

for m in metrics:
    metrics_table = pd.concat([metrics_table, pd.DataFrame({
        "Model": [m["model"]],
        "MAE_Train": [m["mae"][0]],
        "MAE_Valid": [m["mae"][1]],
        "MAE_Test": [m["mae"][2]],
        "RMSE_Train": [m["rmse"][0]],
        "RMSE_Valid": [m["rmse"][1]],
        "RMSE_Test": [m["rmse"][2]],
        "R2_Train": [m["r2"][0]],
        "R2_Valid": [m["r2"][1]],
        "R2_Test": [m["r2"][2]],
    })], ignore_index=True)

metrics_table

,Feature,SHAP Importance
Rank,,
0,bedrooms,449.064238
1,bathrooms,419.125298
2,Doorman,287.969485
3,interest_level,222.445833
4,LaundryinUnit,139.210677
5,FitnessCenter,77.581411
6,Elevator,74.675064
7,DogsAllowed,61.044906
8,Dishwasher,54.181816


/tmp/ipython-input-3110713923.py:18: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  metrics_table = pd.concat([metrics_table, pd.DataFrame({


,Model,MAE_Train,MAE_Valid,MAE_Test,RMSE_Train,RMSE_Valid,RMSE_Test,R2_Train,R2_Valid,R2_Test
0,Lasso MinMaxScaler,694.229071,685.342591,681.668548,1008.221322,973.255097,987.718662,0.604320,0.617410,0.610598
1,Lasso top10 MinMaxScaler,694.307082,685.183373,681.711667,1008.322721,973.262092,987.811224,0.604240,0.617405,0.610525
2,Lasso permutation MinMaxScaler,699.787075,692.365413,685.028143,1017.555679,983.920945,994.276833,0.596959,0.608979,0.605410


In [145]:
perm_df = pd.DataFrame({
    "feature": X_train_num.columns,
    "Mean": perm.importances_mean,
    "Std": perm.importances_std
}).sort_values(by="Mean", ascending=False).head(10)

perm_display = pd.DataFrame({
    "Feature": perm_df["feature"],
    "Mean ± Std Deviation": perm_df.apply(lambda r: f"{r['Mean']:.5f} ± {r['Std']:.3f}", axis=1)
})

perm_display.index.name = "neg_mean_absolute_percentage_error"

perm_display

,Feature,Mean ± Std Deviation
neg_mean_absolute_percentage_error,,
21,bedrooms,0.07769 ± 0.002
20,bathrooms,0.07195 ± 0.001
4,Doorman,0.03088 ± 0.001
22,interest_level,0.02566 ± 0.001
10,LaundryinUnit,0.00940 ± 0.000
8,FitnessCenter,0.00337 ± 0.000
0,Elevator,0.00303 ± 0.000
5,Dishwasher,0.00240 ± 0.000
9,Pre-War,0.00158 ± 0.000


### 7. Hyperparameter optimization

ElasticNet GridSearchCV

In [140]:
param_grid = {"alpha":[0.01,0.1,0.5,1.0], "l1_ratio":[0.1,0.5,0.7,0.9,1.0]}
grid_search = GridSearchCV(ElasticNet(random_state=21,max_iter=20000), param_grid, scoring='neg_mean_absolute_error', cv=3)
grid_search.fit(X_train_s, y_train)
print("Grid Search best params:", grid_search.best_params_)
best_grid_model = grid_search.best_estimator_
metrics.append(evaluate(best_grid_model, X_train_s, X_valid_s, X_test_s, y_train, y_val, y_test, "Grid Search ElasticNet"))

Grid Search best params: {'alpha': 1.0, 'l1_ratio': 1.0}


RandomizedSearchCV

In [141]:
param_dist = {"alpha": uniform(0.01,1.0), "l1_ratio": uniform(0.1,0.9)}
random_search = RandomizedSearchCV(ElasticNet(random_state=21,max_iter=20000), param_distributions=param_dist, n_iter=20, scoring='neg_mean_absolute_error', cv=3, n_jobs=-1, random_state=21)
random_search.fit(X_train_s, y_train)
print("Random Search best params:", random_search.best_params_)
best_random_model = random_search.best_estimator_
metrics.append(evaluate(best_random_model, X_train_s, X_valid_s, X_test_s, y_train, y_val, y_test, "Random Search ElasticNet"))

Random Search best params: {'alpha': np.float64(0.07957095461260054), 'l1_ratio': np.float64(0.8806640355937795)}


Optuna ElasticNet

In [142]:
def objective(trial):
    alpha = trial.suggest_float("alpha", 0.01, 1.0, log=True)
    l1_ratio = trial.suggest_float("l1_ratio", 0.1, 1.0)
    model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, random_state=21, max_iter=20000)
    model.fit(X_train_s, y_train)
    return mean_absolute_error(y_val, model.predict(X_valid_s))

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)
print("Optuna best params:", study.best_params)
optuna_model = ElasticNet(**study.best_params, random_state=21, max_iter=20000)
optuna_model.fit(X_train_s, y_train)
metrics.append(evaluate(optuna_model, X_train_s, X_valid_s, X_test_s, y_train, y_val, y_test, "Optuna ElasticNet"))

[I 2025-11-23 22:48:21,683] A new study created in memory with name: no-name-0085e72c-ce25-4ba1-8405-2cb0b513394b
[I 2025-11-23 22:48:21,740] Trial 0 finished with value: 980.0112781870262 and parameters: {'alpha': 0.4340746726548281, 'l1_ratio': 0.6439074512669174}. Best is trial 0 with value: 980.0112781870262.
[I 2025-11-23 22:48:21,801] Trial 1 finished with value: 896.4675151062498 and parameters: {'alpha': 0.07289818605306025, 'l1_ratio': 0.2991672915546211}. Best is trial 1 with value: 896.4675151062498.
[I 2025-11-23 22:48:21,893] Trial 2 finished with value: 817.8242299017011 and parameters: {'alpha': 0.022348143570808967, 'l1_ratio': 0.1343143499308366}. Best is trial 2 with value: 817.8242299017011.
[I 2025-11-23 22:48:22,177] Trial 3 finished with value: 839.6314045815815 and parameters: {'alpha': 0.133788943296646, 'l1_ratio': 0.8084154474052782}. Best is trial 2 with value: 817.8242299017011.
[I 2025-11-23 22:48:22,202] Trial 4 finished with value: 1056.0220447938352 and 

Optuna best params: {'alpha': 0.010338891272023464, 'l1_ratio': 0.843991993688015}


In [143]:
metrics_table = pd.DataFrame(columns=[
    "Model", "MAE_Train", "MAE_Valid", "MAE_Test",
    "RMSE_Train", "RMSE_Valid", "RMSE_Test",
    "R2_Train", "R2_Valid", "R2_Test"
])

for m in metrics:
    metrics_table = pd.concat([metrics_table, pd.DataFrame({
        "Model": [m["model"]],
        "MAE_Train": [m["mae"][0]],
        "MAE_Valid": [m["mae"][1]],
        "MAE_Test": [m["mae"][2]],
        "RMSE_Train": [m["rmse"][0]],
        "RMSE_Valid": [m["rmse"][1]],
        "RMSE_Test": [m["rmse"][2]],
        "R2_Train": [m["r2"][0]],
        "R2_Valid": [m["r2"][1]],
        "R2_Test": [m["r2"][2]]
    })], ignore_index=True)

metrics_table


/tmp/ipython-input-54178407.py:8: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  metrics_table = pd.concat([metrics_table, pd.DataFrame({


,Model,MAE_Train,MAE_Valid,MAE_Test,RMSE_Train,RMSE_Valid,RMSE_Test,R2_Train,R2_Valid,R2_Test
0,Lasso MinMaxScaler,694.229071,685.342591,681.668548,1008.221322,973.255097,987.718662,0.604320,0.617410,0.610598
1,Lasso top10 MinMaxScaler,694.307082,685.183373,681.711667,1008.322721,973.262092,987.811224,0.604240,0.617405,0.610525
2,Lasso permutation MinMaxScaler,699.787075,692.365413,685.028143,1017.555679,983.920945,994.276833,0.596959,0.608979,0.605410
3,Grid Search ElasticNet,693.994183,685.663795,681.369258,1008.599559,974.073855,987.956828,0.604023,0.616766,0.610410
4,Random Search ElasticNet,776.069429,773.758534,767.014831,1142.706573,1115.505168,1123.177218,0.491721,0.497399,0.496466
5,Optuna ElasticNet,718.377377,714.854738,708.975285,1047.888053,1018.204996,1028.151480,0.572573,0.581254,0.578065


1. Lasso full

Что делает: Обучает модель Lasso (линейная регрессия с L1-регуляризацией) на всех числовых признаках.

Зачем: Позволяет сразу оценить вклад всех признаков и одновременно обнуляет коэффициенты для малозначимых признаков, что даёт простую форму отбора признаков.

2. Lasso top10

Что делает: Обучение Lasso только на топ-10 признаках по абсолютной величине коэффициентов из Lasso full.

Зачем: Сокращает модель до самых значимых признаков, чтобы проверить, насколько сильно они объясняют цену.

3. Permutation importance

Что делает: Случайным образом перемешивает значения каждого признака на валидации и смотрит, как это влияет на метрики (MAE).

Зачем: Позволяет понять реальную важность признака для модели, а не только по величине коэффициента.

Результат: Топ-10 признаков, которые больше всего ухудшают предсказание при перемешивании.

4. SHAP (SHapley Additive exPlanations)

Что делает: Использует теорию Шепли для оценки вклада каждого признака в предсказание для каждой строки данных.

Зачем: Обеспечивает интерпретируемость модели — показывает, какие признаки и как влияют на результат.

Результат: Топ-10 признаков с наибольшим средним абсолютным влиянием (SHAP importance).

5. GridSearchCV

Что делает: Перебирает заранее заданные комбинации гиперпараметров (alpha, l1_ratio) для ElasticNet.

Зачем: Находит наилучшие параметры модели по метрике (MAE) с использованием кросс-валидации.

6. RandomizedSearchCV

Что делает: Случайным образом пробует N комбинаций гиперпараметров из заданного диапазона.

Зачем: Быстрее, чем GridSearch, когда параметров много; позволяет найти хорошие значения без полного перебора.

7. Optuna

Что делает: Использует оптимизацию на основе проб и ошибок (Bayesian optimization) для поиска лучших гиперпараметров ElasticNet.

Зачем: Автоматически подбирает оптимальные alpha и l1_ratio с минимизацией ошибки на валидации.

Результат: Наиболее точная модель среди всех методов поиска гиперпараметров.